In [1]:
import pandas as pd

sample = pd.read_csv("data.csv")
split_df = pd.read_csv("split_labels.csv")

# Convert sample rows to list of dicts
sample_rows = sample.to_dict(orient="records")

# Build nodes + links
nodes = set()
links = {}

for r in sample_rows:
    src = f"{r['from_layer']}_{r['from_cluster']}"
    tgt = f"{r['to_layer']}_{r['to_cluster']}"
    nodes.add(src)
    nodes.add(tgt)

    key = (src, tgt)
    links[key] = links.get(key, 0) + 1

nodes = [{"name": n} for n in sorted(nodes)]
links_out = [{"source": s, "target": t, "value": v} for (s,t), v in links.items()]

# Build split map
split_map_dict = dict(zip(split_df.real_id.astype(str), split_df.split_label))

In [2]:
data_dict = {
    "nodes": nodes,
    "links": links_out,
    "sampleRows": sample_rows,
}

In [3]:
import json, html
from IPython.display import HTML

# 读取 sankey_template.html
with open("sankey_template.html") as f:
    template = f.read()

# 读取 sankey_core.js（必须内联）
with open("sankey_core.js") as f:
    core_js = f.read()

# 这里 data_dict 和 split_map_dict 是你的 Python 数据
DATA_JSON = json.dumps(data_dict)
SPLIT_JSON = json.dumps(split_map_dict)

html_full = template.replace(
    '<script src="sankey_core.js"></script>',
    f"<script>{core_js}</script>"   # 直接内联
).replace(
    "window.__DATA__ = {}",
    f"window.__DATA__ = {DATA_JSON}"
).replace(
    "window.__SPLIT_MAP__ = {}",
    f"window.__SPLIT_MAP__ = {SPLIT_JSON}"
)

escaped = html.escape(html_full)

iframe_html = f"""
<iframe
    srcdoc="{escaped}"
    width="100%" height="650"
    style="border:none;"
></iframe>
"""

HTML(iframe_html)

In [3]:
import os
import json, html
import pandas as pd
from IPython.display import HTML

folder = "scatter_plot_data_by_layer"
layers_dict = {}

for fname in os.listdir(folder):
    if not fname.endswith(".csv"):
        continue
    
    layer_name = fname.replace("scatter_plot_data_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(folder, fname))

    # 期望列: x, y, color, real_id
    layers_dict[layer_name] = df.to_dict(orient="records")

print("Loaded layers:", list(layers_dict.keys()))


with open("scatter_template.html") as f:
    template = f.read()

with open("scatter_core.js") as f:
    core_js = f.read()

LAYER_JSON = json.dumps(layers_dict)

html_full = (
    template
        .replace('<script src="scatter_core.js"></script>',
                 f"<script>{core_js}</script>")   # 内联 JS
        .replace("window.__SCATTER_DATA__ = {};",
                 f"window.__SCATTER_DATA__ = {LAYER_JSON};")
)

escaped = html.escape(html_full)

iframe_html = f"""
<iframe
    srcdoc="{escaped}"
    width="100%" height="650"
    style="border:none;"
></iframe>
"""

HTML(iframe_html)

Loaded layers: ['emb', 'output_block_3', 'output_block_2', 'output_block_0', 'output_block_1', 'output_block_4', 'u_mlp_input', 'u_mlp_l1', 'u_mlp_l2']
